In [2]:
from pathlib import Path
import json

# Find the root directory, no matter where we are. 
def find_project_root(marker="README.md"):
    p = Path.cwd()
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise RuntimeError("Project root not found")

PROJECT_ROOT = find_project_root()

path = PROJECT_ROOT / "notebooks/api_progress.json"

In [3]:
# Read in feature engineered data 

input_path = PROJECT_ROOT / "data/processed/feature_engineered_data.csv"
import pandas as pd
df = pd.read_csv(input_path)

In [4]:
input_path = PROJECT_ROOT / "data/progress_tracking/urls_with_descriptions.csv"
descriptions = pd.read_csv(input_path)

In [5]:
descriptions.rename(columns={'description': 'full_description'}, inplace=True)

In [6]:
df = df.merge(descriptions[['redirect_url', 'full_description']], on='redirect_url', how='left')

In [ ]:
df['full_description']  = df['full_description'].fillna('')
df['full_description_length'] = df['full_description'].str.len()

In [ ]:
X_structured = df.drop(columns=['avg_salary', 'title', 'description'])
X_structured.columns

cat_vars = ['contract_type', 'contract_time', 'category.label', 'location.area_length', 
            # 'location_country', 
            'location_state', 'location_region_abridged', 'location_city_abridged', 'missing_long_lat']

num_vars = ['longitude', 'latitude']

In [10]:
# Model training and validation functions 

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sentence_transformers import SentenceTransformer
import pandas as pd

EMBEDDING_MODEL = SentenceTransformer('all-MiniLM-L6-v2')


def get_text_series(df, text_config):
    """
    Select the appropriate text column from the dataframe.
    text_config: 'title', 'description', or 'title_description'
    """
    valid = ['title', 'description', 'title_description', 'full_description']
    if text_config not in valid:
        raise ValueError(f"text_config must be one of {valid}, got '{text_config}'")
    return df[text_config]


def build_text_features(train_series, val_series, feature_config, reduce=False, n_components=100):
    """
    Fit text transformers on train_series, transform both train and val.
    feature_config: 'tfidf', 'embeddings', or 'both'
    Returns X_train, X_val as numpy arrays.
    """

    def get_tfidf(train, val):
        tfidf = TfidfVectorizer(max_features=10_000, ngram_range=(1, 2), min_df=3, sublinear_tf=True)
        X_train = tfidf.fit_transform(train)
        X_val = tfidf.transform(val)
        if reduce:
            svd = TruncatedSVD(n_components=n_components, random_state=42)
            X_train = svd.fit_transform(X_train)
            X_val = svd.transform(X_val)
        else:
            X_train = X_train.toarray()
            X_val = X_val.toarray()
        return X_train, X_val

    def get_embeddings(train, val):
        X_train = EMBEDDING_MODEL.encode(train.tolist(), batch_size=64, show_progress_bar=False)
        X_val = EMBEDDING_MODEL.encode(val.tolist(), batch_size=64, show_progress_bar=False)
        if reduce:
            pca = PCA(n_components=n_components, random_state=42)
            X_train = pca.fit_transform(X_train)
            X_val = pca.transform(X_val)
        return X_train, X_val

    if feature_config == 'tfidf':
        return get_tfidf(train_series, val_series)
    elif feature_config == 'embeddings':
        return get_embeddings(train_series, val_series)
    elif feature_config == 'both':
        tfidf_train, tfidf_val = get_tfidf(train_series, val_series)
        emb_train, emb_val = get_embeddings(train_series, val_series)
        return np.hstack([tfidf_train, emb_train]), np.hstack([tfidf_val, emb_val])
    else:
        raise ValueError(f"feature_config must be 'tfidf', 'embeddings', or 'both', got '{feature_config}'")


def evaluate(y_true, y_pred):
    """
    Calculate regression metrics.
    Returns dict with rmse, mae, r2.
    """
    return {
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'mae': mean_absolute_error(y_true, y_pred),
        'r2': r2_score(y_true, y_pred)
    }

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
# Reduce down to descriptions longer than 30 characters - we do this for both the non description and description model to use the same dataset for fairness. 
df = df[df['full_description_length'] > 30]

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import numpy as np
from sklearn.impute import SimpleImputer

df = df.reset_index(drop=True)
y = df['avg_salary'].values

# encode categoricals
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
cat_encoded = ohe.fit_transform(df[cat_vars])


imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()

# scale continuous
cont_imputed = imputer.fit_transform(df[num_vars])
cont_scaled = scaler.fit_transform(cont_imputed)


# combine into one non-text feature matrix
X_structured = np.hstack([cat_encoded, cont_scaled])

KeyError: "['location_region_abridged', 'location_city_abridged', 'missing_long_lat'] not in index"

In [ ]:
# Text the performance of a mdoel that reduces description but not title: 
n_splits = 5
use = 'text'
model_type = 'ridge'

kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
results = []


fold_metrics = []

for fold_num, (train_idx, val_idx) in enumerate(kf.split(df)):
    # Print statement so we can track progress more granularly
    print(f"  Fold {fold_num + 1}/{n_splits}")
    # slice text
    title_series = get_text_series(df, 'title')
    description_series = get_text_series(df, 'description')
    
    train_title_text = title_series.iloc[train_idx]
    val_title_text = title_series.iloc[val_idx]
    train_description_text = description_series.iloc[train_idx]
    val_description_text = description_series.iloc[val_idx]

    # build text features
    X_title_train, X_title_val = build_text_features(
        train_title_text, val_title_text,
        feature_config='tfidf',
        reduce= False 
    )
    
    X_description_train, X_description_val = build_text_features(
        train_description_text, val_description_text,
        feature_config='tfidf',
        reduce= True
    )

    # slice and combine with non-text features
    
    if use == 'text':
        X_train = np.hstack([X_title_train, X_description_train])
        X_val = np.hstack([X_title_val, X_description_val])
    elif use == 'structured':
        X_train = X_structured[train_idx]
        X_val = X_structured[val_idx]
    elif use == 'both':
        X_train = np.hstack([X_title_train, X_description_train, X_structured[train_idx]])
        X_val = np.hstack([X_title_val, X_description_val, X_structured[val_idx]])
    else:
        raise ValueError(f"use must be 'text', 'structured', or 'both', got '{use}'")

    y_train = y[train_idx]
    y_val = y[val_idx]

    # fit model
    model = Ridge(alpha=1.0) if model_type == 'ridge' else Lasso(alpha=0.1)
    model.fit(X_train, y_train)

    # evaluate
    y_pred = model.predict(X_val)
    fold_metrics.append(evaluate(y_val, y_pred))

print(fold_metrics)

  Fold 1/5


  Fold 2/5
  Fold 3/5
  Fold 4/5
  Fold 5/5
[{'rmse': np.float64(38827.01240414371), 'mae': 27838.835839360756, 'r2': 0.45947089300726673}, {'rmse': np.float64(37203.01185970801), 'mae': 26409.05975962767, 'r2': 0.44882005760626964}, {'rmse': np.float64(34994.37418455303), 'mae': 25406.707243696677, 'r2': 0.49766098920852175}, {'rmse': np.float64(33215.46722100048), 'mae': 24475.94006661987, 'r2': 0.5675516837633106}, {'rmse': np.float64(36529.668507149145), 'mae': 26590.19588541045, 'r2': 0.48104368338120385}]


In [ ]:
# Test how much the full description adds to predictiive power. 
n_splits = 5
use = 'text'
model_type = 'ridge'

kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
results = []


fold_metrics = []

for fold_num, (train_idx, val_idx) in enumerate(kf.split(df)):
    # Print statement so we can track progress more granularly
    print(f"  Fold {fold_num + 1}/{n_splits}")
    # slice text
    title_series = get_text_series(df, 'title')
    # This is the only line that changes.
    description_series = get_text_series(df, 'full_description')
    
    train_title_text = title_series.iloc[train_idx]
    val_title_text = title_series.iloc[val_idx]
    train_description_text = description_series.iloc[train_idx]
    val_description_text = description_series.iloc[val_idx]

    # build text features
    X_title_train, X_title_val = build_text_features(
        train_title_text, val_title_text,
        feature_config='tfidf',
        reduce= False 
    )
    
    X_description_train, X_description_val = build_text_features(
        train_description_text, val_description_text,
        feature_config='tfidf',
        reduce= True
    )

    # slice and combine with non-text features
    
    if use == 'text':
        X_train = np.hstack([X_title_train, X_description_train])
        X_val = np.hstack([X_title_val, X_description_val])
    elif use == 'structured':
        X_train = X_structured[train_idx]
        X_val = X_structured[val_idx]
    elif use == 'both':
        X_train = np.hstack([X_title_train, X_description_train, X_structured[train_idx]])
        X_val = np.hstack([X_title_val, X_description_val, X_structured[val_idx]])
    else:
        raise ValueError(f"use must be 'text', 'structured', or 'both', got '{use}'")

    y_train = y[train_idx]
    y_val = y[val_idx]

    # fit model
    model = Ridge(alpha=1.0) if model_type == 'ridge' else Lasso(alpha=0.1)
    model.fit(X_train, y_train)

    # evaluate
    y_pred = model.predict(X_val)
    fold_metrics.append(evaluate(y_val, y_pred))

print(fold_metrics)

  Fold 1/5
  Fold 2/5
  Fold 3/5
  Fold 4/5
  Fold 5/5
[{'rmse': np.float64(36685.81453292812), 'mae': 26116.549866625148, 'r2': 0.5174442792416024}, {'rmse': np.float64(35759.43634802936), 'mae': 24503.654520995555, 'r2': 0.49076466251843576}, {'rmse': np.float64(33874.333786612115), 'mae': 24842.051998304265, 'r2': 0.5293024165344916}, {'rmse': np.float64(31424.34901718915), 'mae': 22700.88531656671, 'r2': 0.6129330785715966}, {'rmse': np.float64(35050.80712669593), 'mae': 25045.322704527793, 'r2': 0.5222118368359279}]


In [ ]:
n_splits = 5
use = 'structured'
model_type = 'ridge'

kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
results = []


fold_metrics = []

for fold_num, (train_idx, val_idx) in enumerate(kf.split(df)):
    # Print statement so we can track progress more granularly
    print(f"  Fold {fold_num + 1}/{n_splits}")
    # slice text
    title_series = get_text_series(df, 'title')
    # This is the only line that changes.
    description_series = get_text_series(df, 'full_description')
    
    train_title_text = title_series.iloc[train_idx]
    val_title_text = title_series.iloc[val_idx]
    train_description_text = description_series.iloc[train_idx]
    val_description_text = description_series.iloc[val_idx]

    # slice and combine with non-text features
    
    if use == 'text':
        X_train = np.hstack([X_title_train, X_description_train])
        X_val = np.hstack([X_title_val, X_description_val])
    elif use == 'structured':
        X_train = X_structured[train_idx]
        X_val = X_structured[val_idx]
    elif use == 'both':
        X_train = np.hstack([X_title_train, X_description_train, X_structured[train_idx]])
        X_val = np.hstack([X_title_val, X_description_val, X_structured[val_idx]])
    else:
        raise ValueError(f"use must be 'text', 'structured', or 'both', got '{use}'")

    y_train = y[train_idx]
    y_val = y[val_idx]

    # fit model
    model = Ridge(alpha=1.0) if model_type == 'ridge' else Lasso(alpha=0.1)
    model.fit(X_train, y_train)

    # evaluate
    y_pred = model.predict(X_val)
    fold_metrics.append(evaluate(y_val, y_pred))

print(fold_metrics)

  Fold 1/5
  Fold 2/5
  Fold 3/5
  Fold 4/5
  Fold 5/5
[{'rmse': np.float64(42914.026058411015), 'mae': 31488.97398250572, 'r2': 0.33968728901217604}, {'rmse': np.float64(43081.26261997441), 'mae': 31589.475962142093, 'r2': 0.2608815389043433}, {'rmse': np.float64(39413.628066273144), 'mae': 29834.743761299123, 'r2': 0.3627743170920299}, {'rmse': np.float64(41172.33970256106), 'mae': 30507.034658368542, 'r2': 0.3355465301369688}, {'rmse': np.float64(40944.21241916848), 'mae': 31426.370315576212, 'r2': 0.348034847317368}]


In [ ]:
# Text the performance of a mdoel that reduces description but not title (and includes structured data): 
n_splits = 5
use = 'both'
model_type = 'ridge'

kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
results = []


fold_metrics = []

for fold_num, (train_idx, val_idx) in enumerate(kf.split(df)):
    # Print statement so we can track progress more granularly
    print(f"  Fold {fold_num + 1}/{n_splits}")
    # slice text
    title_series = get_text_series(df, 'title')
    description_series = get_text_series(df, 'description')
    
    train_title_text = title_series.iloc[train_idx]
    val_title_text = title_series.iloc[val_idx]
    train_description_text = description_series.iloc[train_idx]
    val_description_text = description_series.iloc[val_idx]

    # build text features
    X_title_train, X_title_val = build_text_features(
        train_title_text, val_title_text,
        feature_config='tfidf',
        reduce= False 
    )
    
    X_description_train, X_description_val = build_text_features(
        train_description_text, val_description_text,
        feature_config='tfidf',
        reduce= True
    )

    # slice and combine with non-text features
    
    if use == 'text':
        X_train = np.hstack([X_title_train, X_description_train])
        X_val = np.hstack([X_title_val, X_description_val])
    elif use == 'structured':
        X_train = X_structured[train_idx]
        X_val = X_structured[val_idx]
    elif use == 'both':
        X_train = np.hstack([X_title_train, X_description_train, X_structured[train_idx]])
        X_val = np.hstack([X_title_val, X_description_val, X_structured[val_idx]])
    else:
        raise ValueError(f"use must be 'text', 'structured', or 'both', got '{use}'")

    y_train = y[train_idx]
    y_val = y[val_idx]

    # fit model
    model = Ridge(alpha=1.0) if model_type == 'ridge' else Lasso(alpha=0.1)
    model.fit(X_train, y_train)

    # evaluate
    y_pred = model.predict(X_val)
    fold_metrics.append(evaluate(y_val, y_pred))

print(fold_metrics)

  Fold 1/5
  Fold 2/5
  Fold 3/5
  Fold 4/5
  Fold 5/5
[{'rmse': np.float64(36087.10028075435), 'mae': 26165.54267505146, 'r2': 0.533066418437918}, {'rmse': np.float64(36803.546050149635), 'mae': 26207.930076619687, 'r2': 0.4605930543236709}, {'rmse': np.float64(33348.67899939191), 'mae': 25062.473934378904, 'r2': 0.5437974439161282}, {'rmse': np.float64(32696.92862200541), 'mae': 23786.79907197139, 'r2': 0.580948502059088}, {'rmse': np.float64(34667.8614830352), 'mae': 26236.189097835657, 'r2': 0.5325949014939113}]


In [ ]:
# Test how much the full description adds to predictiive power (with structured data). 
n_splits = 5
use = 'both'
model_type = 'ridge'

kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
results = []


fold_metrics = []

for fold_num, (train_idx, val_idx) in enumerate(kf.split(df)):
    # Print statement so we can track progress more granularly
    print(f"  Fold {fold_num + 1}/{n_splits}")
    # slice text
    title_series = get_text_series(df, 'title')
    # This is the only line that changes.
    description_series = get_text_series(df, 'full_description')
    
    train_title_text = title_series.iloc[train_idx]
    val_title_text = title_series.iloc[val_idx]
    train_description_text = description_series.iloc[train_idx]
    val_description_text = description_series.iloc[val_idx]

    # build text features
    X_title_train, X_title_val = build_text_features(
        train_title_text, val_title_text,
        feature_config='tfidf',
        reduce= False 
    )
    
    X_description_train, X_description_val = build_text_features(
        train_description_text, val_description_text,
        feature_config='tfidf',
        reduce= True
    )

    # slice and combine with non-text features
    
    if use == 'text':
        X_train = np.hstack([X_title_train, X_description_train])
        X_val = np.hstack([X_title_val, X_description_val])
    elif use == 'structured':
        X_train = X_structured[train_idx]
        X_val = X_structured[val_idx]
    elif use == 'both':
        X_train = np.hstack([X_title_train, X_description_train, X_structured[train_idx]])
        X_val = np.hstack([X_title_val, X_description_val, X_structured[val_idx]])
    else:
        raise ValueError(f"use must be 'text', 'structured', or 'both', got '{use}'")

    y_train = y[train_idx]
    y_val = y[val_idx]

    # fit model
    model = Ridge(alpha=1.0) if model_type == 'ridge' else Lasso(alpha=0.1)
    model.fit(X_train, y_train)

    # evaluate
    y_pred = model.predict(X_val)
    fold_metrics.append(evaluate(y_val, y_pred))

print(fold_metrics)

  Fold 1/5
  Fold 2/5
  Fold 3/5
  Fold 4/5
  Fold 5/5
[{'rmse': np.float64(34408.604289550756), 'mae': 24591.119214126637, 'r2': 0.5754926150400146}, {'rmse': np.float64(35508.91171986454), 'mae': 24884.503934567252, 'r2': 0.4978749035356892}, {'rmse': np.float64(32255.802966895186), 'mae': 24660.003908383365, 'r2': 0.5732081096800272}, {'rmse': np.float64(30973.65702007849), 'mae': 22958.664232875184, 'r2': 0.6239561859291239}, {'rmse': np.float64(34402.04641931694), 'mae': 25424.016792043014, 'r2': 0.5397350592252275}]


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
# Try a gradient boosting model to see if we have any non-lineaities of interest. 
gb = GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=42)


# Test how much the full description adds to predictiive power (with structured data). 
n_splits = 5
use = 'both'
model_type = 'gb'

kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
results = []


fold_metrics = []

for fold_num, (train_idx, val_idx) in enumerate(kf.split(df)):
    # Print statement so we can track progress more granularly
    print(f"  Fold {fold_num + 1}/{n_splits}")
    # slice text
    title_series = get_text_series(df, 'title')
    # This is the only line that changes.
    description_series = get_text_series(df, 'full_description')
    
    train_title_text = title_series.iloc[train_idx]
    val_title_text = title_series.iloc[val_idx]
    train_description_text = description_series.iloc[train_idx]
    val_description_text = description_series.iloc[val_idx]

    # build text features
    X_title_train, X_title_val = build_text_features(
        train_title_text, val_title_text,
        feature_config='tfidf',
        reduce= False 
    )
    
    X_description_train, X_description_val = build_text_features(
        train_description_text, val_description_text,
        feature_config='tfidf',
        reduce= True
    )

    # slice and combine with non-text features
    
    if use == 'text':
        X_train = np.hstack([X_title_train, X_description_train])
        X_val = np.hstack([X_title_val, X_description_val])
    elif use == 'structured':
        X_train = X_structured[train_idx]
        X_val = X_structured[val_idx]
    elif use == 'both':
        X_train = np.hstack([X_title_train, X_description_train, X_structured[train_idx]])
        X_val = np.hstack([X_title_val, X_description_val, X_structured[val_idx]])
    else:
        raise ValueError(f"use must be 'text', 'structured', or 'both', got '{use}'")

    y_train = y[train_idx]
    y_val = y[val_idx]

    # fit model
    model = gb if model_type == 'gb' else Ridge(alpha=0.1)
    model.fit(X_train, y_train)

    # evaluate
    y_pred = model.predict(X_val)
    fold_metrics.append(evaluate(y_val, y_pred))

print(fold_metrics)

  Fold 1/5
  Fold 2/5
  Fold 3/5
  Fold 4/5
  Fold 5/5
[{'rmse': np.float64(35270.25646232058), 'mae': 23983.496407761897, 'r2': 0.5539655826050789}, {'rmse': np.float64(36499.76500374122), 'mae': 23477.975318225795, 'r2': 0.4694609682543397}, {'rmse': np.float64(29939.114349053532), 'mae': 21194.139188847803, 'r2': 0.632312943850161}, {'rmse': np.float64(29498.438220396678), 'mae': 20763.441824943384, 'r2': 0.6589237141079828}, {'rmse': np.float64(31383.296824640725), 'mae': 22598.340366638124, 'r2': 0.6169667557427354}]
